# Item Update: Point-Estimate vs. Uncertainty-Weighted

Supports appendix subsection *Item update: point-estimate vs. uncertainty-weighted*
(`app:item-update`).

The IXPLORE item step re-fits per-item logistic regressions from the current user
representations. Two variants:

- **Point-estimate** (`use_point_estimates=True`): each user contributes one training
  point at their posterior-mean position.
- **Uncertainty-weighted** (`use_point_estimates=False`): the full per-user grid
  posterior is used as a soft assignment (EM-style M-step).

This reproduces the study of the (removed) `run_posterior_effect.py` with the current
IXPLORE API. As in that script, a single model is fitted per `(sparsity, seed)` with
`n_iterations=1` and `sampling_resolution=100`; the posteriors are computed once and
only the item step is re-run under each variant (`fit_posteriors()` then
`fit_models()`), so the two conditions share the same user posteriors and differ only
in the item update. Results are written incrementally so a timeout cannot discard
completed cells. Output table: `tables/posterior_effect_smartvote_2023.tex`.


In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from src.data import load_dataset
from src.models.ixplore_wrapper import IXPLOREModel

DATASET = "smartvote_2023"
SPARSITY_LEVELS = [0.0, 0.3, 0.6, 0.9]
N_SEEDS = 5
N_ITERATIONS = 1          # matches the original run_posterior_effect.py
SAMPLING_RESOLUTION = 100  # matches the original run_posterior_effect.py

RESULTS_DIR = Path("../../results") / DATASET / "posterior_effect"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR = Path("../../tables")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_CSV = RESULTS_DIR / "train_metrics.csv"
TEST_CSV = RESULTS_DIR / "test_metrics.csv"


## Run both variants across the sparsity/seed grid

For each `(sparsity u, seed)` the model is fitted once; `fit_posteriors()` is called
once; then for each variant we set `use_point_estimates` and call `fit_models()` so the
two conditions share the same user posteriors. Train cells are recorded per `(u, seed)`;
test cells come from the `u = 0` model evaluated across the test-sparsity grid `v`.
Each `(condition, u, seed)` row is appended to CSV immediately (idempotent: rows already
present are skipped on re-run).


In [2]:
dataset = load_dataset(DATASET)

def done_keys(csv, cols):
    if not csv.exists():
        return set()
    d = pd.read_csv(csv)
    if d.empty:
        return set()
    return set(map(tuple, d[cols].itertuples(index=False, name=None)))

def append_row(csv, row):
    pd.DataFrame([row]).to_csv(csv, mode="a", header=not csv.exists(), index=False)

train_done = done_keys(TRAIN_CSV, ["model", "sparsity", "seed"])
test_done = done_keys(TEST_CSV, ["model", "sparsity", "seed"])

for u in tqdm(SPARSITY_LEVELS, desc="train sparsity"):
    train_seeds = range(N_SEEDS) if u > 0 else [0]
    for seed in train_seeds:
        need = [c for c in ("posterior_mean", "posterior_weighted")
                if (c, u, seed) not in train_done]
        need_test = (u == 0.0) and any(
            (c, v, s) not in test_done
            for c in ("posterior_mean", "posterior_weighted")
            for v in SPARSITY_LEVELS
            for s in (range(N_SEEDS) if v > 0 else [0])
        )
        if not need and not need_test:
            continue

        sparse_train = dataset.get_data_with_sparsity("train", u, seed)
        model = IXPLOREModel(n_iterations=N_ITERATIONS,
                             sampling_resolution=SAMPLING_RESOLUTION)
        model.fit(sparse_train)
        model.fit_posteriors()  # user posteriors computed once, shared across conditions

        for condition, point_est in [("posterior_mean", True),
                                     ("posterior_weighted", False)]:
            model.use_point_estimates = point_est
            model._m().use_point_estimates = point_est
            model.fit_models()  # only the item step changes between conditions

            if (condition, u, seed) not in train_done:
                tm = model.evaluate(sparse_train, dataset.train_reactions)
                append_row(TRAIN_CSV, {**tm, "model": condition,
                                       "sparsity": u, "seed": seed})

            if u == 0.0:
                for v in SPARSITY_LEVELS:
                    test_seeds = range(N_SEEDS) if v > 0 else [0]
                    for seed_test in test_seeds:
                        if (condition, v, seed_test) in test_done:
                            continue
                        sparse_test = dataset.get_data_with_sparsity("test", v, seed_test)
                        em = model.evaluate(sparse_test, dataset.test_reactions,
                                            test=True)
                        append_row(TEST_CSV, {**em, "model": condition,
                                              "sparsity": v, "seed": seed_test})

print("train rows:", len(pd.read_csv(TRAIN_CSV)))
print("test rows :", len(pd.read_csv(TEST_CSV)))


train sparsity:   0%|          | 0/4 [00:00<?, ?it/s]

train rows: 32
test rows : 32


## Table: test imputation MAE by item-update variant

One row per test sparsity `v in {0.3, 0.6, 0.9}` (imputation undefined at `v = 0`),
mean +/- std over the 5 seeds, with the uncertainty-weighted minus point-estimate
difference. Exported as `posterior_effect_smartvote_2023.tex` for `app:item-update`.


In [3]:
test_df = pd.read_csv(TEST_CSV)

def mean_mae(cond, v):
    s = test_df[(test_df["model"] == cond) & (test_df["sparsity"] == v)]["impute_mae"].dropna()
    return s.mean()

rows = []
for v in [0.3, 0.6, 0.9]:
    pe = mean_mae("posterior_mean", v)
    uw = mean_mae("posterior_weighted", v)
    rows.append((v, pe, uw, uw - pe))

summary = pd.DataFrame(rows, columns=["sparsity", "pe", "uw", "delta"])
display(summary.round(4))

# Report convention: plain means, best in column bold (no +/- std).
lines = [
    r"\begin{tabular}{cccc}",
    r"\toprule",
    r"Sparsity & Point-estimate & Uncertainty-weighted & $\Delta$ \\",
    r"\midrule",
]
for _, r in summary.iterrows():
    pe_s = f"{r.pe:.4f}"
    uw_s = f"{r.uw:.4f}"
    if r.pe <= r.uw:
        pe_s = r"\textbf{" + pe_s + "}"
    else:
        uw_s = r"\textbf{" + uw_s + "}"
    lines.append(f"{r.sparsity:g} & {pe_s} & {uw_s} & ${r.delta:+.4f}$ \\\\")
lines += [r"\bottomrule", r"\end{tabular}", ""]

out = TABLES_DIR / "posterior_effect_smartvote_2023.tex"
out.write_text("\n".join(lines))
print("wrote", out)
print("\n".join(lines))


,sparsity,pe,uw,delta
0,0.3,0.2522,0.2531,0.0009
1,0.6,0.2553,0.2562,0.0009
2,0.9,0.2743,0.2744,0.0001


wrote ../../tables/posterior_effect_smartvote_2023.tex
\begin{tabular}{cccc}
\toprule
Sparsity & Point-estimate & Uncertainty-weighted & $\Delta$ \\
\midrule
0.3 & \textbf{0.2522} & 0.2531 & $+0.0009$ \\
0.6 & \textbf{0.2553} & 0.2562 & $+0.0009$ \\
0.9 & \textbf{0.2743} & 0.2744 & $+0.0001$ \\
\bottomrule
\end{tabular}

